# 🏃📊 Tracking data (ATD)

The **Automatic Tracking Data**: computer-vision output giving the position and speed of every
tracked object on the pitch, once per video frame, **including the ball**.

This notebook covers loading it, how it lines up with the frames you label, what it does not
contain, and how to look at it.

---

## Loading

The ATD is Metrica's **FIFA/EPTS** export — a metadata XML plus a raw text file — so
[kloppy](https://kloppy.pysport.org/user-guide/loading-data/metrica/) reads the pair directly.

The match was recorded twice. This is the **tactical** recording, the one a submission is
indexed to; `source='broadcast'` gets the other, on its own timeline.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from kloppy import metrica

import games

paths = games.asset_paths(source='tactical')

# The full file is 199,319 frames. Enough here to reach the first-half kickoff at 11185.
dataset = metrica.load_tracking_epts(meta_data=paths['atd_meta'],
                                     raw_data=paths['atd_raw'], limit=12000)
print('%d frames loaded' % len(dataset.records))

## The data as a table

Each row is one video frame:

- **frame_id** — the tracking's own counter, starting at **1**
- **ball_x**, **ball_y** — the ball, when it is visible
- **1_x**, **1_y**, **1_s** — position and speed (m/s) of track slot 1, and so on

All positions are **normalised to 0–1**, not metres. At 25 fps, 25 rows is one second.

In [ ]:
frame_df = dataset.to_df()
frame_df.head()

## Frame alignment — a single subtraction

The ATD counts from 1 and your submission counts from 0:

```
video_frame = frame_id - 1
```

That is the whole alignment story. `load_atd.check_alignment()` verifies it rather than
asserting it — the record count must equal the video length, and the ids must be exactly
`1..n` with nothing missing.

In [ ]:
import load_atd

# Straight off the raw file -- one streaming pass, no kloppy. Loading all 199,319 frames through
# kloppy just to count them would cost minutes and gigabytes.
report = load_atd.check_alignment_raw()
for key, value in report.items():
    print('%-18s %s' % (key, value))

## What it does not contain

### No player identities

There are **46 track slots** for 22 players. They are named `Track 1`…`Track 46` and their
`ShirtNumber` is just the slot index — **not a shirt number, and not a person**. Slots start,
stop and swap identity as the tracker loses and re-acquires people.

You get positions and team membership. Any analysis you build has to be collective.

In [ ]:
players = dataset.metadata.teams[0].players + dataset.metadata.teams[1].players
print('%d track slots' % len(players))
for player in players[:3]:
    print('  %-10s shirt %-3s team %s' % (player.name, player.jersey_no, player.team))
print('  ...')

### The ball is missing 43% of the time

The ball is the most informative signal in the file and the least reliable one. Count it
yourself before building anything that assumes it is there.

In [ ]:
ball = frame_df[['ball_x', 'ball_y']]
seen = ball.notna().all(axis=1)
print('ball present on %d of %d loaded frames (%.1f%%)' % (seen.sum(), len(seen),
                                                           100 * seen.mean()))
print()
print('over the whole match it is 56.9%. See DATA.md.')

## Pitch dimensions

Positions are normalised, so multiply by the real pitch to get metres.

In [ ]:
length = dataset.metadata.pitch_dimensions.pitch_length
width = dataset.metadata.pitch_dimensions.pitch_width
print('ATD metadata: %s x %s m' % (length, width))
print('games.py:     %s x %s m  (measured from the corrected tracking)'
      % games.GAMES[games.GAME]['pitch'])

## Looking at a frame

`starter/utils.py` draws a pitch and plots both teams **and the ball**. The kickoff of the
first half is video frame 11185, which is also the first frame the ground truth grades.

In [ ]:
from utils import plot_frame

plot_frame(dataset, 11185)   # 1st half kickoff
plt.show()

Now a frame where the tracking has lost the ball. The title says so — there is no white dot,
and that is the data being honest rather than a plotting bug.

In [ ]:
missing = int(np.flatnonzero(~seen.values)[len(np.flatnonzero(~seen.values)) // 2])
plot_frame(dataset, missing)
plt.show()

## Watching it move

A static frame hides the thing that actually tells you who has the ball: how the shape moves.

The ball simply disappears on frames where it was not tracked.

In [ ]:
from IPython.display import HTML
from utils import animate_frames

anim = animate_frames(dataset, 11185, 11435)   # 10 seconds from kickoff
HTML(anim.to_jshtml())

---

**Next:** [`03_first_submission.ipynb`](03_first_submission.ipynb) turns this into a scored
submission in one sitting.